# CIC-IDS2017 - preprocessing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadDevX/ML-Research-Dataset-Imbalance/blob/main/cicids2017/00_preprocessing/cicids2017_preprocessing.ipynb)

Stateless steps only: label mapping, removing identifiers and duplicates, marking invalid values
as missing, and (for large data) a fixed stratified sample. Everything that is *fitted*
(imputation, scaling, TF-IDF, resampling) happens later inside each training fold.

Writes `data/processed/cicids2017.parquet` and the fixed 5x2 outer folds `data/splits/cicids2017_outer.npz`.
Both are committed, so every experiment notebook (and your teammate) uses identical data and folds.
Raw data is expected in `data/raw/` (not committed; see `data/README.md`).

In [1]:
# ---- Setup: identical in every notebook. Do not edit. ----
import os, sys, subprocess
REPO_URL = "https://github.com/MuhammadDevX/ML-Research-Dataset-Imbalance.git"
BRANCH = "main"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = "/content/ML-Research-Dataset-Imbalance"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_ROOT}/requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Results go to Google Drive so they survive a disconnect.
    os.environ.setdefault("RESULTS_ROOT", "/content/drive/MyDrive/ML-Research-Dataset-Imbalance")
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_ROOT, "common")):
        parent = os.path.dirname(REPO_ROOT)
        if parent == REPO_ROOT:
            raise RuntimeError("Open this notebook from inside the repository.")
        REPO_ROOT = parent

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from common.env import check_environment
check_environment()


Environment OK: scikit-learn 1.9.1, imbalanced-learn 0.14.2, xgboost 3.4.1, lightgbm 4.7.0


## Pass 1: index every flow (stateless steps only)
The 8 `MachineLearningCVE` CSVs of CIC-IDS2017 (about 2.83M flows, 78 features).
Holding all of them in memory needs more than 4 GB, so this runs in two passes:
pass 1 keeps only a 64-bit hash of each flow's features plus its label, which is
enough to find duplicates and draw the samples; pass 2 re-reads the files and keeps
only the selected flows. Runs locally in about 2 minutes (raw data is not on Colab).

* Strip the leading spaces in column names.
* Drop `Fwd Header Length.1`, an exact copy of `Fwd Header Length`.
* `inf` -> NaN (imputed later inside each training fold).
* Remove duplicate flows (same features and label, across all files), then flows
  whose features appear with both labels.
* Target: BENIGN -> 0, any attack -> 1. The attack name is kept as `meta_label`
  for description only; it is never a feature.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from common import data, settings
from common.paths import RAW_DIR

files = sorted((RAW_DIR / "CIC-IDS2017").glob("*.csv"))
header = pd.read_csv(files[0], nrows=0, encoding="latin-1").columns
label_raw = [c for c in header if c.strip() == "Label"][0]
dtypes = {c: "float64" for c in header if c != label_raw}


def read_clean(f):
    d = pd.read_csv(f, encoding="latin-1", dtype=dtypes)
    d.columns = d.columns.str.strip()
    d = d.drop(columns=["Fwd Header Length.1"]).rename(columns={"Label": "meta_label"})
    # The raw files spell "Web Attack – Brute Force" with a broken dash character.
    d["meta_label"] = (d["meta_label"].astype(str).str.replace("\ufffd", "-", regex=False)
                       .str.replace("ï¿½", "-", regex=False).str.strip())
    feats = d.columns.drop("meta_label")
    n_inf = int(np.isinf(d[feats].to_numpy()).sum())
    d[feats] = d[feats].mask(np.isinf(d[feats]))
    return d, list(feats), n_inf


index, n_inf = [], 0
for i, f in enumerate(files):
    d, feat_cols, k = read_clean(f)
    n_inf += k
    index.append(pd.DataFrame({
        "file": i, "row": np.arange(len(d)),
        "fhash": pd.util.hash_pandas_object(d[feat_cols], index=False).to_numpy(),
        "label": d["meta_label"].to_numpy()}))
    print(f"{f.name}: {len(d):,} flows")
    del d
index = pd.concat(index, ignore_index=True)
print(f"raw flows: {len(index):,}, inf values: {n_inf:,}, features: {len(feat_cols)}")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 flows


Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 flows


Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 flows


Monday-WorkingHours.pcap_ISCX.csv: 529,918 flows


Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 flows


Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 flows


Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 flows


Wednesday-workingHours.pcap_ISCX.csv: 692,703 flows
raw flows: 2,830,743, inf values: 4,376, features: 77


In [3]:
n_raw = len(index)
index = index[~index.duplicated(subset=["fhash", "label"])]
print(f"duplicate flows removed: {n_raw - len(index):,}")
conflict = index["fhash"].duplicated(keep=False)
print(f"flows with conflicting labels removed: {int(conflict.sum()):,}")
index = index[~conflict].reset_index(drop=True)
index["y"] = (index["label"] != "BENIGN").astype(int)
print(f"after cleaning: {len(index):,} flows, attack rate {index['y'].mean():.2%}")
index["label"].value_counts()

duplicate flows removed: 308,381


flows with conflicting labels removed: 1,396
after cleaning: 2,520,966 flows, attack rate 16.87%


label
BENIGN                        2095786
DoS Hulk                       172719
DDoS                           128013
PortScan                        90255
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5384
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

## Select the rows
* **Main dataset:** stratified 50,000-flow sample; keeps the natural attack rate.
* **E3 pool (BENIGN vs Bot):** all Bot flows plus 10,000 random BENIGN flows.

Both samples are fixed by `MASTER_SEED`.

In [4]:
main_idx, _ = train_test_split(np.arange(len(index)), train_size=50_000,
                               stratify=index["y"], random_state=settings.MASTER_SEED)
main_sel = index.iloc[np.sort(main_idx)]
bot_sel = index[index["label"] == "Bot"]
benign_sel = index[index["label"] == "BENIGN"].sample(n=settings.E3_N, random_state=settings.MASTER_SEED)
e3_sel = pd.concat([benign_sel, bot_sel])
print(f"main: {len(main_sel):,} flows; E3 pool: {len(benign_sel):,} BENIGN + {len(bot_sel):,} Bot")

main: 50,000 flows; E3 pool: 10,000 BENIGN + 1,953 Bot


## Pass 2: read the selected flows and save

In [5]:
def collect(sel):
    """Read the selected (file, row) flows, returned in the order of `sel`."""
    parts = []
    for i, f in enumerate(files):
        rows = sel.loc[sel["file"] == i, "row"].to_numpy()
        if len(rows):
            d, _, _ = read_clean(f)
            part = d.iloc[rows]
            part.index = pd.MultiIndex.from_arrays([np.full(len(rows), i), rows])
            parts.append(part)
            del d
    order = pd.MultiIndex.from_arrays([sel["file"].to_numpy(), sel["row"].to_numpy()])
    out = pd.concat(parts).loc[order].reset_index(drop=True)
    out["y"] = (out["meta_label"] != "BENIGN").astype(int)
    return out[feat_cols + ["meta_label", "y"]]


main = collect(main_sel)
data.save_processed("cicids2017", main)
data.make_outer_splits(main["y"], "cicids2017")
print(main["meta_label"].value_counts())
data.summary("cicids2017")

meta_label
BENIGN                        41567
DoS Hulk                       3437
DDoS                           2552
PortScan                       1731
DoS GoldenEye                   239
FTP-Patator                     122
DoS slowloris                   114
DoS Slowhttptest                103
SSH-Patator                      61
Bot                              37
Web Attack - Brute Force         22
Web Attack - XSS                 13
Web Attack - Sql Injection        2
Name: count, dtype: int64


{'dataset': 'cicids2017',
 'n': 50000,
 'd': 77,
 'n_minority': 8433,
 'minority_pct': 16.866,
 'IR': 4.93,
 'nan_cells': 70}

## E3 pool: BENIGN vs Bot at controlled imbalance
Each level uses N = 10,000 rows with minority share [0.15, 0.1, 0.05, 0.02, 0.01].
Levels are nested (the rows of a smaller minority share are a subset of the larger
one), so only the imbalance changes between levels.

In [6]:
rng = np.random.default_rng(settings.MASTER_SEED)
pool = collect(e3_sel)
data.save_processed("cicids2017_e3", pool)

pos = rng.permutation(np.flatnonzero(pool["y"] == 1))
neg = rng.permutation(np.flatnonzero(pool["y"] == 0))
for level in settings.E3_LEVELS:
    n_min = round(level * settings.E3_N)
    assert n_min <= len(pos), f"not enough Bot flows for level {level}"
    subset = np.sort(np.concatenate([pos[:n_min], neg[:settings.E3_N - n_min]]))
    data.make_outer_splits(pool["y"].to_numpy()[subset], "cicids2017_e3", level=level, subset=subset)
    print(f"level {level:.0%}: {n_min} Bot + {settings.E3_N - n_min} BENIGN")

level 15%: 1500 Bot + 8500 BENIGN
level 10%: 1000 Bot + 9000 BENIGN
level 5%: 500 Bot + 9500 BENIGN


level 2%: 200 Bot + 9800 BENIGN
level 1%: 100 Bot + 9900 BENIGN
